<a href="https://colab.research.google.com/github/petitoandrea-jpg/freeCodeCamp-Machine-Learning-Python/blob/main/sms_classifier/sms_text_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# 1. Carica i file TSV nei DataFrame
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['type', 'text'])
test_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['type', 'text'])

# 2. Converti le etichette in valori numerici: ham = 0, spam = 1
train_df['label'] = train_df['type'].map({'ham': 0, 'spam': 1})
test_df['label'] = test_df['type'].map({'ham': 0, 'spam': 1})

train_text = train_df['text'].values
train_labels = train_df['label'].values
test_text = test_df['text'].values
test_labels = test_df['label'].values

# 3. Preparazione del layer di TextVectorization
VOCAB_SIZE = 10000
MAX_LEN = 100

vectorize_layer = keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
vectorize_layer.adapt(train_text)

# 4. Definizione del modello
model = keras.Sequential([
    vectorize_layer,
    keras.layers.Embedding(VOCAB_SIZE, 64),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 5. Addestramento del modello
model.fit(
    train_text,
    train_labels,
    epochs=10,
    batch_size=32,
    validation_data=(test_text, test_labels),
    verbose=1
)

In [ ]:
def predict_message(pred_text):
    # Convertiamo la stringa in un Tensor di TensorFlow per compatibilità con Keras 3
    input_text = tf.constant([pred_text])

    # Calcola la probabilità (da 0 a 1)
    prob = model.predict(input_text, verbose=0)[0][0]

    # Assegna l'etichetta in base alla soglia 0.5
    label = "spam" if prob >= 0.5 else "ham"

    return [float(prob), label]

pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()